# QM 640 Capstone — Step 1: EDGAR Event Identification

Enterprise AI Investment and Stock Market Performance: An Event Study

Searches SEC EDGAR full-text search for 8-K filings that mention AI-related
investment/partnership/R&D/M&A activity.

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [2]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 890, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 890 (delta 35), reused 44 (delta 19), pack-reused 813 (from 1)
Receiving objects: 100% (890/890), 5.11 MiB | 22.25 MiB/s, done.
Resolving deltas: 100% (476/476), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [3]:
!pip install -q requests pandas

## Cell 3 — Configuration

**Edit `USER_AGENT` below before running** — SEC rejects requests without a
real name and email in this header.

In [4]:
import os

# ---------------------------------------------------------------------
# CONFIG - edit before running
# ---------------------------------------------------------------------
USER_AGENT = "Shanmuganathan Ekambaram QM640 Capstone Shan_muganathan@yahoo.com"  # SEC REQUIRES a real contact

OUTPUT_DIR = os.path.join(BASE_DIR, "data/raw")
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "edgar_candidate_events.csv")

# Keyword set from Synopsis "Event identification" section
KEYWORDS = [
    '"artificial intelligence investment"',
    '"AI partnership"',
    '"generative AI"',
    '"AI acquisition"',
    '"AI research and development"',
    # added — broader phrasing, same underlying concept
    '"artificial intelligence partnership"',
    '"AI collaboration"',
    '"AI initiative"',
    '"machine learning investment"',
    '"strategic AI"',
    '"generative artificial intelligence"',
    '"AI technology investment"',
    '"large language model"',
    # added — widened after the reclassification pass showed the original list
    # under-collecting relative to the N=159 minimum (confirmed 118 vs 181 reviewed)
    '"AI-powered"',
    '"AI-driven"',
    '"AI capabilities"',
    '"AI solution"',
    '"AI-enabled"',
    '"responsible AI"',
    '"applied AI"',
    '"enterprise AI"',
    '"conversational AI"',
    '"predictive AI"',
    '"AI model"',
    '"deep learning investment"',
    '"AI-based"',
]

START_DATE = "2023-01-01"
END_DATE = "2026-08-01"   # widened from 2026-07-25 - update to today's date when you actually run this
FORM_TYPES = "8-K"

SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
HEADERS = {"User-Agent": USER_AGENT}

print("Will write output to:", OUTPUT_FILE)

Will write output to: /content/QM640-WALSH-CAPSTONE/data/raw/edgar_candidate_events.csv


## Cell 4 — EDGAR search function

In [5]:
import requests
import time
import csv


def search_edgar(query, start_date, end_date, forms="8-K"):
    """Query EDGAR full-text search, paginating through all hits (10,000 cap)."""
    results = []
    frm = 0
    page_size = 100

    while True:
        params = {
            "q": query,
            "forms": forms,
            "dateRange": "custom",
            "startdt": start_date,
            "enddt": end_date,
            "from": frm,
        }
        resp = requests.get(SEARCH_URL, params=params, headers=HEADERS, timeout=20)
        resp.raise_for_status()
        data = resp.json()

        hits = data.get("hits", {}).get("hits", [])
        if not hits:
            break

        for h in hits:
            src = h.get("_source", {})
            results.append({
                "query": query,
                "cik": src.get("ciks", [None])[0],
                "company_name": src.get("display_names", [None])[0],
                "form_type": src.get("root_forms", [None])[0],
                "file_date": src.get("file_date"),
                "accession_no": h.get("_id"),
                "adsh": src.get("adsh"),
                "file_name": src.get("file_name") or src.get("_id"),
            })

        total = data.get("hits", {}).get("total", {}).get("value", 0)
        frm += page_size
        if frm >= total or frm >= 9900:  # EDGAR caps at 10,000 results
            break

        time.sleep(0.15)  # stay under 10 req/sec

    return results

## Cell 5 — Run the search and save results

In [6]:
all_rows = []
seen_accessions = set()

for kw in KEYWORDS:
    print(f"Searching EDGAR for: {kw} ...")
    rows = search_edgar(kw, START_DATE, END_DATE, FORM_TYPES)
    print(f"  -> {len(rows)} hits")
    for r in rows:
        if r["accession_no"] not in seen_accessions:
            seen_accessions.add(r["accession_no"])
            all_rows.append(r)
    time.sleep(0.5)

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "query", "cik", "company_name", "form_type",
        "file_date", "accession_no", "adsh", "file_name"
    ])
    writer.writeheader()
    writer.writerows(all_rows)

print(f"\nTotal unique candidate filings: {len(all_rows)}")
print(f"Saved to: {OUTPUT_FILE}")

Searching EDGAR for: "artificial intelligence investment" ...
  -> 0 hits
Searching EDGAR for: "AI partnership" ...
  -> 24 hits
Searching EDGAR for: "generative AI" ...
  -> 2045 hits
Searching EDGAR for: "AI acquisition" ...
  -> 130 hits
Searching EDGAR for: "AI research and development" ...
  -> 18 hits
Searching EDGAR for: "artificial intelligence partnership" ...
  -> 2 hits
Searching EDGAR for: "AI collaboration" ...
  -> 21 hits
Searching EDGAR for: "AI initiative" ...
  -> 27 hits
Searching EDGAR for: "machine learning investment" ...
  -> 3 hits
Searching EDGAR for: "strategic AI" ...
  -> 40 hits
Searching EDGAR for: "generative artificial intelligence" ...
  -> 695 hits
Searching EDGAR for: "AI technology investment" ...
  -> 0 hits
Searching EDGAR for: "large language model" ...
  -> 196 hits
Searching EDGAR for: "AI-powered" ...
  -> 3923 hits
Searching EDGAR for: "AI-driven" ...
  -> 3766 hits
Searching EDGAR for: "AI capabilities" ...
  -> 983 hits
Searching EDGAR for: 

## Commit and push results back to GitHub

In [7]:
!git -C {BASE_DIR} add "data/raw/edgar_candidate_events.csv"
!git -C {BASE_DIR} commit -m "Step 1: EDGAR candidate event search"
!git -C {BASE_DIR} push

[main 1efef25] Step 1: EDGAR candidate event search
 1 file changed, 10850 insertions(+)
 create mode 100644 data/raw/edgar_candidate_events.csv
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 342.47 KiB | 4.96 MiB/s, done.
Total 5 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   fe49852..1efef25  main -> main


## Sanity check

In [8]:
import pandas as pd

df = pd.read_csv(OUTPUT_FILE)
print("Shape:", df.shape)
df.head()

Shape: (10849, 8)


,query,cik,company_name,form_type,file_date,accession_no,adsh,file_name
0,"""AI partnership""",1830214,"Ginkgo Bioworks Holdings, Inc. (DNA, DNA-WT) ...",8-K,2023-11-08,0001628280-23-037916:ex-991q32023earnings.htm,0001628280-23-037916,NaN
1,"""AI partnership""",1830214,"Ginkgo Bioworks Holdings, Inc. (DNA, DNA-WT) ...",8-K,2023-08-29,0000950170-23-044892:dna-ex99_1.htm,0000950170-23-044892,NaN
2,"""AI partnership""",1830214,"Ginkgo Bioworks Holdings, Inc. (DNA, DNA-WT) ...",8-K,2023-08-29,0000950170-23-044892:dna-20230829.htm,0000950170-23-044892,NaN
3,"""AI partnership""",1570585,"Liberty Global Ltd. (LBTYA, LBTYB, LBTYK) (C...",8-K,2026-02-03,0001570585-26-000004:ex991lgandgooglecloudanno...,0001570585-26-000004,NaN
4,"""AI partnership""",1824502,"Archer Aviation Inc. (ACHR, ACHR-WT) (CIK 00...",8-K,2025-05-12,0001628280-25-024657:archershareholderletter-.htm,0001628280-25-024657,NaN
